<img src="../../../On-Site-Round/Lost_in_Hyperspace/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (Burgas, Bulgarie), epreuve sur site](https://ioai-official.org/bulgaria-2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2024/blob/main/fr/On-Site-Round/Lost_in_Hyperspace/Lost_in_Hyperspace.ipynb)

# Perdu dans l'hyperespace : defi de regression en machine learning

<img src="../../../On-Site-Round/Lost_in_Hyperspace/figs/Lost in Hyperspace Fig 1.png" width="300">

## Contexte narratif
Felicitations pour votre promotion au rang de detective principal en ingenierie ! Votre excellent travail sur la tache precedente vous a valu ce nouveau defi passionnant.
Vous etes maintenant charge des anciens et fascinants hypercubes lumineux, qui partagent certaines caracteristiques intrigantes avec les widgets "Pulse of the Machine" de votre mission precedente (voir la section Conseils importants pour plus de details).
Votre mission consiste a percer les mysteres de ces hypercubes lumineux en predisant trois proprietes essentielles a partir des donnees fournies.
## Objectif et limites
- Votre objectif final est de predire efficacement trois proprietes des hypercubes lumineux.
- Chaque hypercube lumineux est represente par un tableau de forme (5 x 5 x 5 x 6) avec de nombreuses symetries et proprietes uniques (voir la section Conseils importants).
- Vous devez concevoir un petit nombre de caracteristiques a partir des donnees des hypercubes lumineux, car les procedures industrielles efficaces vous autorisent a **utiliser uniquement une regression lineaire** comme modele, sans modification des hyperparametres. Vous etes egalement limite a 300 caracteristiques par tache.
- Votre reussite sera mesuree par l'erreur quadratique moyenne racine (RMSE) pour chaque propriete separement, puis convertie en score sur le leaderboard.
- Les differentes proprietes ont des poids differents dans le score final. Consultez la variable `SCALING_WEIGHTS` pour plus de details. Apres mise a l'echelle, nous ferons la moyenne des RMSE normalisees pour obtenir un score unique.
- Votre solution pour chaque tache ne doit pas depasser 5 minutes pour la generation de caracteristiques, l'entrainement et l'inference sur une instance Colab standard sans GPU.
- Partagez avec nous les fichiers `ml_feature_0.txt`, `ml_feature_1.txt` et `ml_feature_2.txt`, et n'oubliez pas de fournir egalement votre Google Colab.

## Conseils importants

<img src="../../../On-Site-Round/Lost_in_Hyperspace/figs/Lost in Hyperspace Fig 2.png" width="600">

- Documentation de la regression lineaire
  - https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

- Fonctions NumPy utiles :
  - https://numpy.org/doc/stable/reference/generated/numpy.swapaxes.html
  - https://numpy.org/doc/stable/reference/generated/numpy.ravel.html
  - https://numpy.org/doc/stable/reference/generated/numpy.reshape.html

- Erreur quadratique moyenne racine
  - https://en.wikipedia.org/wiki/Root_mean_square_deviation

## Clarification sur l'utilisation des methodes

Les limitations d'utilisation des methodes ressemblent principalement a celles de la tache a domicile :

- Respectez les limites de temps. Elles sont separees pour chaque modele de prevision et supposent une instance sans GPU. Ce temps inclut la generation de caracteristiques, l'entrainement et l'inference sur l'ensemble de validation/test. L'analyse des donnees, la recherche et la selection des caracteristiques ne sont pas soumises a cette limite de temps.

- Les reseaux de neurones supervises (et tout modele supervise : LDA, arbres boosting, etc.) ne sont pas autorises comme extracteurs de caracteristiques. L'utilisation de modeles supervises plus simples (par exemple des ensembles d'arbres ou une regression lineaire) pour la selection de caracteristiques est autorisee, tant que le modele n'est pas utilise comme extracteur de caracteristiques. L'apprentissage non supervise est autorise, y compris les autoencodeurs.

- L'utilisation de modeles preentraines ou de solutions AutoML n'est pas autorisee. Les bibliotheques qui parcourent automatiquement differentes approches pour l'utilisateur ne sont pas autorisees non plus.

- Compte tenu des contraintes de temps, les notebooks Colab doivent etre aussi reproductibles que possible. En cas de doute (abus des limites de temps, utilisation des donnees, etc.), le jury se reserve le droit d'utiliser les modeles/reponses generes par le notebook et de retenir les reponses avec le score le plus faible.

- Oui, differents modeles peuvent utiliser differents ensembles de caracteristiques.

- Non, vous ne pouvez pas utiliser les donnees de validation pour l'entrainement.

**En cas de doute, demandez au jury !**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

SCALING_WEIGHTS = [100/15, 100/8, 100/100]

In [ ]:
data = pd.read_pickle('../../../On-Site-Round/Lost_in_Hyperspace/training_set/ml_data_onsite_start.pickle')
for key in data.keys():
  print(key)

In [ ]:
for key in data['X'].keys():
  print(key)

In [ ]:
for key in data['y'].keys():
  print(key)

In [ ]:
X_train = data['X']['train']
y_train = data['y']['train']

X_val = data['X']['val']
y_val = data['y']['val']

X_test = data['X']['live_test']

In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

In [ ]:
def vis(arr):
  plt.figure(figsize=(8, 8))

  cnt = 1
  for z in range(5):
    for q in range(6):
      plt.subplot(5, 6, cnt)
      plt.imshow(arr[:, :, z, q], vmin=-40, vmax=40, cmap='hsv')
      plt.grid()
      plt.axis('off')
      cnt += 1
  plt.tight_layout()

In [ ]:
vis(X_train[0])

## Fonctions d'evaluation des resultats / d'ecriture des predictions

Ne les modifiez pas !

In [ ]:
def test_solution(X_train, y_train, X_val, y_val, feature_num=0):
    assert X_train.shape[-1] <= 300, "Too many features! Should be less than 300"
    assert X_val.shape[-1] <= 300, "Too many features! Should be less than 300"

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_val)
    rmse = mean_squared_error(
        predictions,
        y_val[:, feature_num]
    )**.5
    normalized_rmse = rmse * SCALING_WEIGHTS[feature_num]
    print(f"Property #{feature_num}:    raw RMSE={rmse:.6f}")
    print(f"Property #{feature_num}: scaled RMSE={normalized_rmse:.6f}")
    return round(normalized_rmse, 6)

## Essayons une solution de reference

In [ ]:
def dummy_feature_extractor(X):
    X_new = X.reshape((X.shape[0], -1)) # mise a plat
    X_new = X_new[:, :300] # conserver les 300 premieres caracteristiques
    return X_new

In [ ]:
 dummy_feature_extractor(X_train).shape

In [ ]:
%%time
total_score = 0
for feature_number in range(3):
  total_score += test_solution(
      dummy_feature_extractor(X_train),
      y_train,
      dummy_feature_extractor(X_val),
      y_val,
      feature_num=feature_number
  )
  print()
total_score /= 3
print('='*16)
print(f"Total score = {total_score:.6f}")

## Comment preparer les fichiers de reponse

In [ ]:
def generate_predictions(X_train, y_train, X_test, feature_num=0):
    assert X_train.shape[-1] <= 300
    assert X_test.shape[-1] <= 300

    model =  LinearRegression().fit(
        X_train,
        y_train[:, feature_num]
    )
    predictions = model.predict(X_test)
    return predictions


## Generer les solutions et les ecrire dans le fichier
combined = {'ID': np.arange(X_test.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('predictions.csv', index=False)

In [ ]:
# charger le jeu de donnees de test
loaded = pd.read_pickle("../../../On-Site-Round/Lost_in_Hyperspace/Solution/test_set/ml_data_onsite_final_test.pickle")
X_test_final = loaded['X']['final_test']


# produire les predictions finales
combined = {'ID': np.arange(X_test_final.shape[0])}

for feature_number in range(3):
    predictions = generate_predictions(
        dummy_feature_extractor(X_train),
        y_train,
        dummy_feature_extractor(X_test_final),
        feature_num=feature_number
    )

    combined[f'y{feature_number+1}'] = predictions

pd.DataFrame(combined).to_csv('final_predictions.csv', index=False)